## Import Library dan Data Loading

    Pada tahap ini, saya melakukan import library, dan data loading dari 'P2M3_Risyadhana_data_clean.csv'

In [1]:
import pandas as pd
import great_expectations as gx
import warnings
import logging

# Mute warnings dari python
warnings.filterwarnings('ignore')

# Mute logging berlebihan dari Great Expectations
logging.getLogger("great_expectations").setLevel(logging.ERROR)

In [2]:
# opsional: Mute progress bar dari GX
# library untuk menonaktifkan progress bar dari GX
'''
import os
os.environ["CHECKPOINT_DISABLE_PROGRESS_BAR"] = "True"
os.environ["GX_DISABLE_PROGRESS_BAR"] = "True"
'''

'\nimport os\nos.environ["CHECKPOINT_DISABLE_PROGRESS_BAR"] = "True"\nos.environ["GX_DISABLE_PROGRESS_BAR"] = "True"\n'

In [3]:
# Load data
df = pd.read_csv('P2M3_Risyadhana_data_clean.csv')

# Inisialisasi context buat dapetin great_expectations
context = gx.get_context()

# Buat expectation suite baru, dengan nama "p2m3_suite"
suite_name = "p2m3_suite"

# Dilakukan dengan try-except untuk menghindari error 
try:
    context.suites.add(gx.ExpectationSuite(name=suite_name))
except:
    pass 

# Setup DataSource & Asset
datasource = context.data_sources.add_pandas(name="my_datasource")
data_asset = datasource.add_dataframe_asset(name="my_asset")

# Build Batch Request
batch_request = data_asset.build_batch_request(options={"dataframe": df})

# Inisialisasi Validator dengan Batch Request dan Expectation Suite yang sudah dibuat
gdf = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=suite_name
)

# print hasil validasi
results = gdf.validate()
print(results)



Calculating Metrics: 0it [00:00, ?it/s]

{
  "success": true,
  "results": [],
  "suite_name": "p2m3_suite",
  "suite_parameters": {},
  "statistics": {
    "evaluated_expectations": 0,
    "successful_expectations": 0,
    "unsuccessful_expectations": 0,
    "success_percent": null
  },
  "meta": {
    "great_expectations_version": "1.12.3",
    "expectation_suite_name": "p2m3_suite",
    "run_id": {
      "run_name": null,
      "run_time": "2026-02-23T17:14:34.255070+07:00"
    },
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260223T101434.247977Z",
      "pandas_data_fingerprint": "a50c2e9660a5ec68316a974a5703a9b4"
    },
    "active_batch_definition": {
      "datasource_name": "my_datasource",
      "data_connector_name": "fluent",
      "data_asset_name": "my_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_time": "20260223T101434.255070Z",
    "checkpoint_name": null
  },
  "id": null
}


## 1. Expectation: Unique

    Pada tahap ini, saya ingin memastikan kombinasi invoice_id dan branch unik untuk setiap transaksi, sehingga nantinya bisa digunakan sebagai id_transaksi_audit yang unik untuk setiap transaksi

In [4]:
# inisiasi kolom baru untuk audit, gabungan antara invoice_id dan branch
df['id_transaksi_audit'] = df['invoice_id'] + "-" + df['branch']

# inisiasi ulang batch request dengan dataframe yang sudah ditambahkan kolom baru
batch_request_updated = data_asset.build_batch_request(options={"dataframe": df})

# Panggil validator lagi dengan batch yang baru
gdf = context.get_validator(
    batch_request=batch_request_updated,
    expectation_suite_name=suite_name
)

# run expectation baru untuk cek kolom id_transaksi_audit
gdf.expect_column_values_to_be_unique('id_transaksi_audit')

Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 359.57it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_unique",
    "kwargs": {
      "batch_id": "my_datasource-my_asset",
      "column": "id_transaksi_audit"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 1000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 2. Expectation: Between (Rating)

    Di data set saya, biasanya rating biasanya skala 1-10 atau 1-100. Sehingga say aset aja di rentang 0 sampai 100 supaya aman.

In [5]:
# Skenario: Memastikan skor kepuasan pelanggan (rating) berada di rentang yang valid (0 sampai 10)
# Jika data kamu skalanya 0-100, silakan ganti max_value menjadi 100
gdf.expect_column_values_to_be_between('rating', min_value=0, 
    max_value=100, # Saya naikkan ke 100 agar lebih mencakup skala umum
    mostly=0.8)

Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 342.93it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "my_datasource-my_asset",
      "column": "rating",
      "mostly": 0.8,
      "min_value": 0.0,
      "max_value": 100.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 1000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 3. Expectation: To be in Set (Gender)

    Pada tahap ini, saya ingin memastikan tidak ada kategori selain Male dan Female, sehingga jika ada data yang tidak sesuai dengan kategori tersebut, bisa dibilang akan dianggap sebagai data yang tidak valid

In [6]:
# inisiasi expectation baru untuk cek kolom gender
gdf.expect_column_values_to_be_in_set('gender', ['Male', 'Female'])

Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 159.77it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_in_set",
    "kwargs": {
      "batch_id": "my_datasource-my_asset",
      "column": "gender",
      "value_set": [
        "Male",
        "Female"
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 1000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 4. Expectation: To be in Type List (Quantity)

    Pada tahap ini, saya ingin memastikan tipe data quantity adalah numerik, sehingga bisa dipastikan bahwa kolom quantity tidak mengandung nilai non-numerik yang bisa menyebabkan error saat analisis lebih lanjut

In [7]:
# inisiasi expectation baru untuk cek kolom quantity, harus berupa angka (int atau float)
gdf.expect_column_values_to_be_in_type_list('quantity', ['int64', 'float64'])

Calculating Metrics: 100%|██████████| 1/1 [00:00<00:00, 198.65it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_in_type_list",
    "kwargs": {
      "batch_id": "my_datasource-my_asset",
      "column": "quantity",
      "type_list": [
        "int64",
        "float64"
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "observed_value": "int64"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 5. Expectation: Column to Exist (Luar Lecture)

    Selanjutnya pada tahap ini, saya ingin memastikan juga bahwa kolom 'invoice_id' ada di dataset, karena ini penting untuk analisis penjualan

In [8]:
# inisiasi expectation baru untuk cek kolom total
gdf.expect_column_to_exist('invoice_id')

Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 188.50it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_to_exist",
    "kwargs": {
      "batch_id": "my_datasource-my_asset",
      "column": "invoice_id"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {},
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 6. Expectation: Column Values to Not be Null (Luar Lecture)

    Pada tahap ini, saya ingin memastikan kolom 'date' tidak ada yang kosong

In [9]:
# inisiasi expectation baru untuk cek kolom date
gdf.expect_column_values_to_not_be_null('date')

Calculating Metrics: 100%|██████████| 6/6 [00:00<00:00, 375.69it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_not_be_null",
    "kwargs": {
      "batch_id": "my_datasource-my_asset",
      "column": "date"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 1000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 7. Expectation: Column Value Lengths (Luar Lecture)

    Pada tahap terakhir ini, saya ingin memastikan jumlah baris berada di antara 1 dan 1 juta (sesuaikan dengan ekspektasi ukuran dataset)

In [ ]:
# inisiasi expectation baru untuk cek kolom
gdf.expect_table_row_count_to_be_between(min_value=1, max_value=1000000)

Calculating Metrics: 100%|██████████| 1/1 [00:00<00:00, 140.99it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_table_row_count_to_be_between",
    "kwargs": {
      "batch_id": "my_datasource-my_asset",
      "min_value": 1,
      "max_value": 1000000
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "observed_value": 1000
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}